# Waveform Propagation Examples

This notebook demonstrates the new architecture for scalar field waveform propagation through galactic density profiles.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from utils.constants import SEC_TO_INEV
from utils.data_utils import read_medium_data, interpolate_data
from inputs.spectrum_sources import (CSVSpectrum, AnalyticSpectrum, SpectrumSource,
                                      TimeVaryingSpectrum, plot_spectrum)
from inputs.configs import PhysicsConfig, PropagationConfig
from utils.logging_utils import setup_logging, get_logger
from waveform.collection import WaveformCollection
from waveform.propagation import plot_spectrogram, export_source_parameters
from utils.constants import KPC_TO_INEV

# Enable inline plotting
%matplotlib inline

# Configure logging
setup_logging(log_file='propagation.log', level='INFO')
logger = get_logger()

## Example 1: Single Static Gaussian

Demonstrates a single Gaussian wavepacket propagating through the galactic density profile.

In [ ]:
from utils.constants import SOLAR_TO_EV

# --- Example 1 Parameters ---
mass = 1e-19                     # Scalar field mass [eV]
N_periods = 100                 # Burst duration in oscillation periods
Etot = SOLAR_TO_EV               # Total energy [eV]
peak_momentum_factor = 2e2       # Peak momentum as multiple of mass
width_factor = 2e1                # Gaussian width as multiple of mass
num_points = 1000                # Spectrum resolution
coupling = 1e20                  # Dilatonic coupling strength
K = 1e-3                         # Energy density fraction
density_num_points = 2000        # Density profile resolution
N_points_spectrogram = 2000      # Spectrogram time steps
param_filename = 'gaussian.param'

In [ ]:
burst_duration = 2 * np.pi * N_periods / (mass * SEC_TO_INEV)
delta_p = width_factor * mass
amplitude = Etot / (np.sqrt(2*np.pi) * delta_p)
spectrum = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=num_points,
    peak_momentum=peak_momentum_factor * mass,
    width=delta_p,
    amplitude=amplitude
)

# Optionally plot the initial spectrum before propagation
plot_spectrum(spectrum, filename='plots/gaussian_initial.pdf')
print(">>> Initial spectrum saved: plots/gaussian_initial.pdf")

# Configure physics and propagation
physics = PhysicsConfig(mass=mass, coupling=coupling, K=K, burst_duration=burst_duration)
propagation_config = PropagationConfig('data/Galactic_Density_Profile.csv', density_num_points=density_num_points)

# Create collection and propagate
collection = WaveformCollection(spectrum, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram)

print(">>> Waveform saved: plots/waveform_plot.pdf")

# Plot spectrogram
R_kpc = (collection.density_profile[0][-1] - collection.density_profile[0][0]) / KPC_TO_INEV
avg_density, f_avg, std_f, arrival_window = plot_spectrogram(
    N_points_spectrogram, results['t_min'], results['t_max'],
    results['E'], results['spectrogram'],
    filename='plots/spectrogram_single_gaussian.pdf',
    show=True,
    mass=mass,
    burst_duration=burst_duration,
    distance=R_kpc
)
print(">>> Spectrogram saved: plots/spectrogram_single_gaussian.pdf")

logger.info("Single Gaussian propagation complete. Spectrogram saved.")

# Export source parameters for constraint plotting workflow
R = collection.density_profile[0][-1] - collection.density_profile[0][0]
export_source_parameters(avg_density, f_avg, std_f, burst_duration, R, mass, arrival_window=arrival_window,
                         to_file=True, filename=param_filename)
print(f">>> Source parameters exported: {param_filename}")

## Example 2: Time-Varying Gaussian

Demonstrates a Gaussian wavepacket with time-varying amplitude modulation.

In [ ]:
# --- Example 2 Parameters ---
mass = 1e-19                     # Scalar field mass [eV]
N_periods = 100                  # Burst duration in oscillation periods
Etot = SOLAR_TO_EV               # Total energy [eV]
width_factor = 20                # Gaussian width as multiple of mass
N_time_steps = 20                # Number of time steps for amplitude modulation
coupling = 1e20                  # Dilatonic coupling strength
K = 1e-3                         # Energy density fraction
density_num_points = 2000        # Density profile resolution
N_points_spectrogram = 2000      # Spectrogram time steps
param_filename = 'gaussian_tv.param'

In [ ]:
burst_duration = 2 * np.pi * N_periods / (mass * SEC_TO_INEV)
delta_p = width_factor * mass
amplitude = Etot / (np.sqrt(2*np.pi) * delta_p)
base_gaussian = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=delta_p,
    amplitude=amplitude
)

# Create time-varying wrapper with Gaussian amplitude profile
amplitude_profile = [np.exp(-(i - N_time_steps/2)**2 / (N_time_steps/3)**2)
                     for i in range(N_time_steps)]

# Define time windows for each step
time_windows = [(i * burst_duration * SEC_TO_INEV / N_time_steps, (i + 1) * burst_duration * SEC_TO_INEV / N_time_steps)
                for i in range(N_time_steps)]

time_varying = TimeVaryingSpectrum(base_gaussian, amplitude_profile, time_windows)

# Configure and propagate
physics = PhysicsConfig(mass=mass, coupling=coupling, K=K, burst_duration=burst_duration)
propagation_config = PropagationConfig('data/Galactic_Density_Profile.csv', density_num_points=density_num_points)

collection = WaveformCollection(time_varying, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
R_kpc = (collection.density_profile[0][-1] - collection.density_profile[0][0]) / KPC_TO_INEV
avg_density, f_avg, std_f, arrival_window = plot_spectrogram(
    N_points_spectrogram, results['t_min'], results['t_max'],
    results['E'], results['spectrogram'],
    filename='plots/spectrogram_time_varying.pdf',
    show=True,
    mass=mass,
    burst_duration=burst_duration,
    distance=R_kpc
)
print(">>> Spectrogram saved: plots/spectrogram_time_varying.pdf")

logger.info(f"Time-varying propagation complete with {N_time_steps} steps")

# Export source parameters for constraint plotting workflow
R = collection.density_profile[0][-1] - collection.density_profile[0][0]
export_source_parameters(avg_density, f_avg, std_f, burst_duration, R, mass, arrival_window=arrival_window,
                         to_file=True, filename=param_filename)
print(f">>> Source parameters exported: {param_filename}")

## Example 3: CSV Spectrum (Bosenova)

Demonstrates loading a spectrum from a CSV file (boson star collapse spectrum).

In [ ]:
from utils.constants import KPC_TO_INEV, GCM3_TO_EV4

# --- Example 3 Parameters ---
mass = 1e-19                      # Scalar field mass [eV]
burst_duration_factor = 400       # Burst duration: factor / (mass * SEC_TO_INEV)
amplitude_normalization = 0.3 / 1e-85  # Amplitude scaling for bosenova spectrum
distance_scale_factor = 1 / 10000  # Distance scaling for density profile (10 kpc → 1 pc)
coupling = 1e22                   # Dilatonic coupling strength
K = 1e-3                          # Energy density fraction
density_num_points = 1000         # Density profile resolution
N_points_spectrogram = 1000       # Spectrogram time steps
param_filename = 'bosenova.param'

In [ ]:
burst_duration = burst_duration_factor / (mass * SEC_TO_INEV)

# Load bosenova spectrum from CSV with scaling applied directly
spectrum = CSVSpectrum(
    'data/BosonStarSpectrumRelOnly.txt',
    i_p=0, 
    i_A=1, 
    skip_header=False, 
    num_points=1000,
    scaled_momentum=mass,
    scaled_amplitude=lambda A: A * amplitude_normalization
)

physics = PhysicsConfig(mass=mass, coupling=coupling, K=K, burst_duration=burst_duration)
propagation_config = PropagationConfig('data/Galactic_Density_Profile.csv', density_num_points=density_num_points)

collection = WaveformCollection(spectrum, physics, propagation_config)

# Override the density profile to apply bosenova distance scaling
x, rho = read_medium_data('data/Galactic_Density_Profile.csv', i_R=0, i_rho=2)
x_interp, rho_interp = interpolate_data(x * distance_scale_factor, rho, density_num_points)
collection.density_profile = [x_interp * KPC_TO_INEV, rho_interp * GCM3_TO_EV4]

results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
R_kpc = (collection.density_profile[0][-1] - collection.density_profile[0][0]) / KPC_TO_INEV
avg_density, f_avg, std_f, arrival_window = plot_spectrogram(
    N_points_spectrogram, results['t_min'], results['t_max'],
    results['E'], results['spectrogram'],
    filename='plots/spectrogram_bosenova.pdf',
    show=True,
    mass=mass,
    burst_duration=burst_duration,
    distance=R_kpc
)
print(">>> Spectrogram saved: plots/spectrogram_bosenova.pdf")

logger.info("CSV (Bosenova) spectrum propagation complete. Spectrogram saved.")

# Export source parameters for constraint plotting workflow
R = collection.density_profile[0][-1] - collection.density_profile[0][0]
export_source_parameters(avg_density, f_avg, std_f, burst_duration, R, mass, arrival_window=arrival_window,
                         to_file=True, filename=param_filename)
print(f">>> Source parameters exported: {param_filename}")

## Summary

This notebook demonstrates the main use cases of the waveform propagation architecture:

1. **Single static spectrum**: Basic Gaussian wavepacket
2. **Time-varying spectrum**: Amplitude modulation over time
3. **CSV spectrum**: Loading external data (bosenova)

Each example produces:
- Initial spectrum plot (optional)
- Waveform plot showing φ(t) at Earth
- Spectrogram showing frequency vs. time evolution
- Parameter file (`.param`) for use with the constraint plotting workflow